# 01 — Load Gold market

End-to-end sanity check for the data loader on the Gold contract files. Replicates spec Figure 1 (1-min bucket volume per contract over time).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from order_mgmt.loader import MarketSpec, load_market

DATA_DIR = Path('..') / 'Order Management' / 'data' / 'Gold'
GOLD = MarketSpec(name='Gold', tick_size=0.10)

In [ ]:
df = load_market(DATA_DIR, GOLD)
print(f'rows: {len(df):,}')
print(f'date range: {df["time"].min()} → {df["time"].max()}')
print(f'contracts present after roll: {sorted(df["contract"].unique().tolist())}')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for symbol, sub in df.groupby('contract'):
    ax.plot(sub['time'], sub['volume'], label=symbol, linewidth=0.5)
ax.set_title('Gold — 1-min volume per active contract')
ax.set_xlabel('date')
ax.set_ylabel('volume')
ax.legend()
plt.tight_layout()

### Sanity checks

- Rows are monotonic in time (no lookahead at the loader boundary).
- Each date is owned by exactly one contract (roll is daily, volume-based).
- Days kept have ≥ 90% of 480 expected trading minutes.

In [ ]:
assert df['time'].is_monotonic_increasing, 'lookahead at loader boundary'
per_date_contracts = df.groupby(df['time'].dt.date)['contract'].nunique()
assert (per_date_contracts == 1).all(), 'a date is owned by more than one contract'
per_date_count = df.groupby(df['time'].dt.date).size()
assert (per_date_count >= 0.90 * 480).all(), 'a kept day has < 90% of expected minutes'
print('all loader-boundary invariants hold')